In [0]:
catalog_name = "workspace"
schema_name = "spotify_pipeline"
silver_table = "spotifytracks_silver"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

spark.sql("SELECT current_catalog(), current_schema()").display()

current_catalog(),current_schema()
workspace,spotify_pipeline


In [0]:
%sql
CREATE OR REPLACE TABLE spotifytracks_gold AS 
(
    WITH PopularityGroups AS (
        SELECT percentile_approx(popularity, 0.10) AS p10,
               percentile_approx(popularity, 0.90) AS p90
        FROM spotifytracks_silver 
    ),
    Grouped AS (
        SELECT CASE WHEN popularity <= p.p10 THEN "low"
        WHEN popularity >= p.p90 THEN "high"
        ELSE "mid"
        END AS popularity_group,
        s.danceability,
        s.energy,
        s.loudness,
        s.acousticness,
        s.valence,
        s.tempo
        FROM PopularityGroups p CROSS JOIN spotifytracks_silver AS s
        )
    SELECT 
        popularity_group,
        AVG(danceability) AS avg_danceability,
        AVG(energy) AS avg_energy,
        AVG(loudness) AS avg_loudness,
        AVG(acousticness) AS avg_acousticness,
        AVG(valence) AS avg_valence,
        AVG(tempo) AS avg_tempo
    FROM Grouped
    WHERE popularity_group != "mid"
    GROUP BY popularity_group
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM spotifytracks_gold; 

popularity_group,avg_danceability,avg_energy,avg_loudness,avg_acousticness,avg_valence,avg_tempo
high,0.6014045561871447,0.6546103899781371,-7.186973939658941,0.25509390535811105,0.49202793528640143,120.8762568430258
low,0.5737242883895125,0.6148779280024945,-8.216587952559266,0.34052273188326976,0.506835422596752,118.87487078651736
